# Introduction to Advanced RAG in LlamaIndex

In [2]:
%pip install nest_asyncio

Note: you may need to restart the kernel to use updated packages.


In [3]:
import nest_asyncio
nest_asyncio.apply()

In [4]:
%pip install -Uq llama-index

Note: you may need to restart the kernel to use updated packages.


## Extract

In [5]:
from llama_index.core import SimpleDirectoryReader

docs = SimpleDirectoryReader(input_dir="./../rag_data").load_data()

# file name as id
# docs_nam_as_id = SimpleDirectoryReader(input_dir="./../rag_data", filename_as_id=True).load_data()

In [6]:
len(docs)  # one per page

16

In [7]:
import pprint
pprint.pprint(docs)

[Document(id_='7a733ff0-4a37-4165-b483-1162112a97be', embedding=None, metadata={'page_label': '1', 'file_name': 'data.pdf', 'file_path': '/home/miria/dev/projects/miria-api/reference/../rag_data/data.pdf', 'file_type': 'application/pdf', 'file_size': 1601210, 'creation_date': '2025-04-16', 'last_modified_date': '2025-04-16'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='SP 103\nFlorida Vegetable Gardening Guide1\nSydney Park Brown, Danielle Treadwell, J. M. Stephens, and Susan Webb2\n1. This document is SP 103, one of a series of the Horticultural Sciences Department, UF/IFAS Extension. Original publication date December 1999. \nRevised

## Transform

In [8]:
# hide some keys from llm

docs[1].__dict__ # too much data about one doc

{'id_': '7655a7d6-70ee-44e5-80db-2b0c790649d4',
 'embedding': None,
 'metadata': {'page_label': '2',
  'file_name': 'data.pdf',
  'file_path': '/home/miria/dev/projects/miria-api/reference/../rag_data/data.pdf',
  'file_type': 'application/pdf',
  'file_size': 1601210,
  'creation_date': '2025-04-16',
  'last_modified_date': '2025-04-16'},
 'excluded_embed_metadata_keys': ['file_name',
  'file_type',
  'file_size',
  'creation_date',
  'last_modified_date',
  'last_accessed_date'],
 'excluded_llm_metadata_keys': ['file_name',
  'file_type',
  'file_size',
  'creation_date',
  'last_modified_date',
  'last_accessed_date'],
 'relationships': {},
 'metadata_template': '{key}: {value}',
 'metadata_separator': '\n',
 'text_resource': MediaResource(embeddings=None, data=None, text='2\nFlorida Vegetable Gardening Guide\nSoil Preparation\nGardeners often plant on whatever soil type is available, \nbut it is usually worthwhile to improve the garden plot with \nadditions of organic matter (see b

In [9]:
# quick example of what the LLM and Embeddings see when with a test document

from llama_index.core import Document
from llama_index.core.schema import MetadataMode

document = Document(
    text="This is a super-customized document",
    metadata={
        "file_name": "super_secret_document.txt",
        "category": "finance",
        "author": "LlamaIndex",
    },
    # excluded_embed_metadata_keys=["file_name"],
    excluded_llm_metadata_keys=["category"],
    metadata_seperator="\n",
    metadata_template="{key}:{value}",
    text_template="Metadata:\n{metadata_str}\n-----\nContent:\n{content}",
)

print(
    "The LLM sees this: \n",
    document.get_content(metadata_mode=MetadataMode.LLM),
)
# print(
#     "The Embedding model sees this: \n",
#     document.get_content(metadata_mode=MetadataMode.EMBED),
# )

The LLM sees this: 
 Metadata:
file_name:super_secret_document.txt
author:LlamaIndex
-----
Content:
This is a super-customized document


In [10]:
from llama_index.core.schema import MetadataMode

# print(docs[0].get_content(metadata_mode=MetadataMode.LLM))   # what the llm sees
print(docs[0].get_content(metadata_mode=MetadataMode.EMBED)) # what embeddings see. in this case, same thing

page_label: 1
file_path: /home/miria/dev/projects/miria-api/reference/../rag_data/data.pdf

SP 103
Florida Vegetable Gardening Guide1
Sydney Park Brown, Danielle Treadwell, J. M. Stephens, and Susan Webb2
1. This document is SP 103, one of a series of the Horticultural Sciences Department, UF/IFAS Extension. Original publication date December 1999. 
Revised October 2015, January 2016, May 2018, September 2020, and September 2021. Visit the EDIS website at https://edis.ifas.ufl.edu for the 
currently supported version of this publication.
2. Sydney Park Brown, associate professor emerita, Environmental Horticulture Department, and adjunct professor, Center for Landscape Conservation 
and Ecology; Danielle Treadwell, assistant professor, Horticultural Sciences Department, and organic farming specialist; J. M. Stephens, professor 
emeritus, Horticultural Sciences Department; and Susan Webb, associate professor, Entomology and Nematology Department; UF/IFAS Extension, 
Gainesville, FL 3261

In [11]:
for doc in docs:
    # define the content/metadata template
    doc.text_template = "Metadata:\n{metadata_str}\n---\nContent:\n{content}"

    # exclude page label from embedding
    if "page_label" not in doc.excluded_embed_metadata_keys:
        doc.excluded_embed_metadata_keys.append("page_label")

In [12]:
# after editing the content seen by embedings

print(docs[0].get_content(metadata_mode=MetadataMode.EMBED))

Metadata:
file_path: /home/miria/dev/projects/miria-api/reference/../rag_data/data.pdf
---
Content:
SP 103
Florida Vegetable Gardening Guide1
Sydney Park Brown, Danielle Treadwell, J. M. Stephens, and Susan Webb2
1. This document is SP 103, one of a series of the Horticultural Sciences Department, UF/IFAS Extension. Original publication date December 1999. 
Revised October 2015, January 2016, May 2018, September 2020, and September 2021. Visit the EDIS website at https://edis.ifas.ufl.edu for the 
currently supported version of this publication.
2. Sydney Park Brown, associate professor emerita, Environmental Horticulture Department, and adjunct professor, Center for Landscape Conservation 
and Ecology; Danielle Treadwell, assistant professor, Horticultural Sciences Department, and organic farming specialist; J. M. Stephens, professor 
emeritus, Horticultural Sciences Department; and Susan Webb, associate professor, Entomology and Nematology Department; UF/IFAS Extension, 
Gainesville,

Here are other, more advanced transformations. Some require an LLM to work. We will use Qwen 2.5 32B Instruct 128k through Groq, which is an affordble, high-rate model. It should be enough to extract Q&As and titles from the documents.

In [13]:
from llama_index.llms.groq import Groq
import os
import getpass


/home/miria/.local/share/virtualenvs/miria-api-sE_hgmYs/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
llm_transformations = Groq(model="llama-3.3-70b-versatile", api_key=os.getenv("GROQ_API_KEY"))

In [15]:
# other transformations

from llama_index.core.extractors import (
    TitleExtractor,
    QuestionsAnsweredExtractor,
)
from llama_index.core.node_parser import SentenceSplitter

text_splitter = SentenceSplitter(
    separator=" ", chunk_size=1024, chunk_overlap=128
)
title_extractor = TitleExtractor(llm=llm_transformations, nodes=5)
qa_extractor = QuestionsAnsweredExtractor(llm=llm_transformations, questions=3)


from llama_index.core.ingestion import IngestionPipeline

pipeline = IngestionPipeline(
    transformations=[
        text_splitter,
        title_extractor,
        qa_extractor
    ]
)

nodes = pipeline.run(
    documents=docs,
    in_place=True,
    show_progress=True,
)

 29%|██▊       | 6/21 [00:11<00:33,  2.20s/it]Retrying llama_index.llms.openai.base.OpenAI._achat in 1.0 seconds as it raised RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01jjn8pp3pewwsca8s5p6wfb2b` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 11911, Requested 1138. Please try again in 5.245s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}.
Retrying llama_index.llms.openai.base.OpenAI._achat in 1.8210248478039237 seconds as it raised RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01jjn8pp3pewwsca8s5p6wfb2b` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 11657, Requested 865. Please try again in 2.609s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/

By default, Llamaindex uses OpenAI's embedding models. But you can choose to load a free model from HuggingFace too (but it it will be slower).

In [16]:
len(nodes)

21

In [17]:
import pprint

# pprint.pprint(nodes[0].__dict__)

print(nodes[0].get_content(metadata_mode=MetadataMode.LLM))

[Excerpt from document]
page_label: 1
file_path: /home/miria/dev/projects/miria-api/reference/../rag_data/data.pdf
document_title: "Comprehensive Guide to Florida Vegetable Gardening: Benefits, Planning, and Best Practices"
questions_this_excerpt_can_answer: Based on the provided context, here are three questions that this context can provide specific answers to, which are unlikely to be found elsewhere:

1. **What are the key factors to consider when selecting a site for a vegetable garden in Florida, and how can I ensure optimal growing conditions?**

This question can be answered by referring to the section "Site" in the excerpt, which provides guidance on locating the garden near the house, choosing a well-drained site, and ensuring at least six hours of direct sunlight daily.

2. **How can I determine the best planting dates for specific vegetables in my area of Florida, and what resources are available to help me plan my garden?**

This question can be answered by referring to th

## Index

In [2]:
# Embeddings

from llama_index.embeddings.huggingface import HuggingFaceEmbedding


hf_embeddings = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

test_embed = hf_embeddings.get_text_embedding("Hello world")


print(test_embed)

[0.015196100808680058, -0.022570667788386345, 0.0085471011698246, -0.07417059689760208, 0.0038364154752343893, 0.0027135491836816072, -0.0312679260969162, 0.04463403671979904, 0.044055208563804626, -0.007871134206652641, -0.025200756266713142, -0.033366620540618896, 0.014427922666072845, 0.04653818905353546, 0.008555104956030846, -0.016145728528499603, 0.007405802607536316, -0.01901242695748806, -0.114726223051548, -0.01815761812031269, 0.12635929882526398, 0.02970289997756481, 0.025281012058258057, -0.034217868000268936, -0.04099970683455467, 0.006617335136979818, 0.010270599275827408, 0.022362269461154938, 0.004436342045664787, -0.12730959057807922, -0.0161492470651865, -0.020380133762955666, 0.047212108969688416, 0.011579900048673153, 0.0681871548295021, 0.007298617158085108, -0.017852986231446266, 0.04078212380409241, -0.010269463062286377, 0.023757092654705048, 0.01060289703309536, -0.028584439307451248, 0.00815972313284874, -0.015180555172264576, 0.0308962594717741, -0.0659798905

In [19]:
# create index

from llama_index.core import VectorStoreIndex

index = VectorStoreIndex(nodes, embed_model=hf_embeddings)

## Query

In [20]:
llm_querying = Groq(model="llama-3.3-70b-versatile", api_key=os.getenv("GROQ_API_KEY"))

query_engine = index.as_query_engine(llm=llm_querying)
response = query_engine.query(
    "what does this model do?"
)

print(response)

This model is an AI-powered study aid that provides a user-friendly interface for interacting with the AI assistant. It delivers a functional retrieval-augmented generation (RAG)-based study aid that achieves at least 70% accuracy in retrieving relevant material and generates study tools rated 4/5 or higher by users. The model also aims to provide evidence of improved comprehension, with at least 60% of testers reporting better understanding of their materials post-use. Additionally, it includes features such as a study space for uploading and analyzing academic materials, study aids like summarization, quiz generation, and flashcard creation, and context-aware recommendations tied to material content.


In [21]:
response.__dict__

{'response': 'This model is an AI-powered study aid that provides a user-friendly interface for interacting with the AI assistant. It delivers a functional retrieval-augmented generation (RAG)-based study aid that achieves at least 70% accuracy in retrieving relevant material and generates study tools rated 4/5 or higher by users. The model also aims to provide evidence of improved comprehension, with at least 60% of testers reporting better understanding of their materials post-use. Additionally, it includes features such as a study space for uploading and analyzing academic materials, study aids like summarization, quiz generation, and flashcard creation, and context-aware recommendations tied to material content.',
 'source_nodes': [NodeWithScore(node=TextNode(id_='fb6ed86d-4ce3-43a7-8d53-56fe5c3d4668', embedding=None, metadata={'page_label': '4', 'file_name': 'thesis.pdf', 'file_path': '/home/miria/dev/projects/miria-api/reference/../rag_data/thesis.pdf', 'file_type': 'application/

## Store

In [22]:
index.storage_context.persist(persist_dir="./vectors")

In [23]:
from llama_index.core import StorageContext, load_index_from_storage

# rebuild storage context
storage_context = StorageContext.from_defaults(persist_dir="./vectors")

# load index
index_from_storage = load_index_from_storage(storage_context, embed_model=hf_embeddings)

In [24]:
qa = index_from_storage.as_query_engine(llm=llm_querying)

In [25]:
response = qa.query("what does this model do?")
print(response)

This model is an AI-powered study aid that provides a user-friendly interface for interacting with the AI assistant. It delivers a functional retrieval-augmented generation (RAG) based study aid that achieves at least 70% accuracy in retrieving relevant material and generates study tools rated 4/5 or higher by users. The model also aims to provide evidence of improved comprehension, with at least 60% of testers reporting better understanding of their materials post-use. Additionally, it includes features such as a study space for uploading and analyzing academic materials, study aids like summarization, quiz generation, and flashcard creation, and context-aware recommendations tied to material content.


# Using Vector Stores

In [26]:
import chromadb
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

# initialize client, setting path to save data
db = chromadb.PersistentClient(path="./chroma_db")

# create collection
chroma_collection = db.get_or_create_collection("healthGPT")

# assign chroma as the vector_store to the context
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# create your index
index = VectorStoreIndex(
    nodes, storage_context=storage_context, embed_model=hf_embeddings
)

# You can also load from documents and apply transformations in place
# index = VectorStoreIndex.from_documents(
#     documents, storage_context=storage_context, transformations=[]
# )

# Or you can initialize your index from your vector store and then add the nodes
# index = VectorStoreIndex.from_vector_store(
#     vector_store=vector_store, embed_model=hf_embeddings
# )
# index.insert_nodes(nodes)


# create a query engine and query
query_engine = index.as_query_engine(llm=llm_querying)

In [32]:
response = query_engine.query("can you generate 20 review cards about the proposal?")
print(response)

Here are 20 review cards about the proposal:

1. **Front**: What is the main goal of the Amu project?
   **Back**: To develop an AI-powered study assistant for CCS students.

2. **Front**: What are the key features of the Amu project?
   **Back**: AI-powered study space, study aids, and context-aware recommendations.

3. **Front**: What is the target audience of the Amu project?
   **Back**: CCS students seeking a smart, content-focused study aid.

4. **Front**: How will the Amu project be evaluated?
   **Back**: Through user testing with 10-25 CCS students and metrics such as accuracy and self-reported comprehension.

5. **Front**: What are the limitations of the Amu project?
   **Back**: Offline functionality may be limited, and precision in retrieving multilingual content may vary.

6. **Front**: What is the expected outcome of the Amu project?
   **Back**: A user-friendly interface, a functional RAG-based study aid, and evidence of improved comprehension.

7. **Front**: What resour